# Function calling

In [3]:
from openai import OpenAI
import json
import os

# Initialize OpenAI client
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")  # Set your API key as environment variable
)

# Step 1: Define a simple function
def get_weather(city):
    """Simple mock weather function"""
    weather_data = {
        "paris": "22°C, sunny",
        "london": "15°C, cloudy", 
        "tokyo": "28°C, rainy",
        "new york": "18°C, partly cloudy"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

# Step 2: Define the function schema for the model (now called "tools")
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a specified city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Step 3: Send message to model with available functions
def chat_with_functions(user_message):
    print(f"User: {user_message}")
    
    # Initial call to the model
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": user_message}],
        tools=tools,
        tool_choice="auto"  # Let model decide when to call functions
    )
    
    message = response.choices[0].message
    
    # Check if model wants to call a function
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        print(f"Model wants to call: {function_name} with args: {function_args}")
        
        # Execute the function
        if function_name == "get_weather":
            function_result = get_weather(function_args["city"])
            print(f"Function result: {function_result}")
            
            # Send function result back to model
            second_response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "user", "content": user_message},
                    message,  # The function call message
                    {
                        "role": "tool",
                        "content": function_result,
                        "tool_call_id": tool_call.id
                    }
                ]
            )
            
            final_answer = second_response.choices[0].message.content
            print(f"Assistant: {final_answer}")
            return final_answer
    else:
        # No function call, just return the regular response
        print(f"Assistant: {message.content}")
        return message.content



In [4]:
# Step 4: Test it
# Example 1: Should trigger function call
chat_with_functions("What's the weather like in Paris?")

print("\n" + "="*50 + "\n")

# Example 2: Should NOT trigger function call
chat_with_functions("Hello, write a short limerick!")

User: What's the weather like in Paris?
Model wants to call: get_weather with args: {'city': 'Paris'}
Function result: 22°C, sunny
Assistant: The weather in Paris is currently 22°C and sunny.


User: Hello, write a short limerick!
Assistant: There once was a user so bright,
Who asked for a limerick just right,
With words that did chime,
In rhythm and rhyme,
Their request brought joy and delight!


'There once was a user so bright,\nWho asked for a limerick just right,\nWith words that did chime,\nIn rhythm and rhyme,\nTheir request brought joy and delight!'

# Calling the MCP Server for US Census data 

## MCP Client functions

In [5]:
import json
import sys
import os
import pandas as pd
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Main function to fetch Census data from MCP server
async def fetch_census_data_async(
    dataset="acs5", 
    year=2022, 
    variables=["B01003_001E"], 
    geography="state", 
    geography_filter=None
):
    """
    Fetch Census data and return as JSON that can be converted to DataFrame.
    
    Args:
        dataset: Census dataset (e.g., 'acs5', 'acs1')
        year: Data year
        variables: List of variable codes
        geography: Geography level
        geography_filter: Optional geography filters
    
    Returns:
        dict: Contains 'success', 'data', and 'error' keys
    """
    
    # Check for API key
    api_key = os.getenv("CENSUS_API_KEY")
    if not api_key:
        return {
            "success": False,
            "error": "CENSUS_API_KEY environment variable not set!",
            "data": None
        }
    
    # Get server script path
    server_script = "us_census_server.py"  # Adjust if needed
    
    # Server parameters
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[server_script],
        env={
            **os.environ,
            "CENSUS_API_KEY": api_key
        }
    )
    
    try:
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                
                # Initialize session
                await session.initialize()
                
                # Tool arguments
                tool_args = {
                    "dataset": dataset,
                    "year": year,
                    "variables": variables,
                    "geography": geography
                }
                
                if geography_filter:
                    tool_args["geography_filter"] = geography_filter
                
                # Call the tool
                result = await session.call_tool("us_census_tool", tool_args)
                
                # Extract the data from new MCP response format
                response_text = None
                
                # The result is now a CallToolResult object
                # We need to iterate through the tuple structure
                for item in result:
                    if isinstance(item, tuple) and len(item) == 2:
                        key, value = item
                        if key == 'content' and isinstance(value, list):
                            # Look for TextContent in the list
                            for content_item in value:
                                if hasattr(content_item, 'text'):
                                    response_text = content_item.text
                                    break
                            break
                
                if not response_text:
                    return {
                        "success": False,
                        "error": "No text content found in server response",
                        "data": None
                    }
                
                # Check if it's an error
                if response_text.startswith("Error"):
                    return {
                        "success": False,
                        "error": response_text,
                        "data": None
                    }
                
                # Extract JSON data from the response
                try:
                    if "Full Data (JSON):" in response_text:
                        json_start = response_text.find("Full Data (JSON):") + len("Full Data (JSON):")
                        json_data = response_text[json_start:].strip()
                        
                        # Parse the JSON
                        data = json.loads(json_data)
                        
                        return {
                            "success": True,
                            "data": data,
                            "error": None,
                            "summary": response_text.split("Full Data (JSON):")[0].strip()
                        }
                    else:
                        # If no JSON found, return the raw response
                        return {
                            "success": True,
                            "data": response_text,
                            "error": None,
                            "summary": response_text
                        }
                except json.JSONDecodeError as e:
                    return {
                        "success": False,
                        "error": f"Failed to parse JSON data: {e}",
                        "data": response_text
                    }
                
    except Exception as e:
        return {
            "success": False,
            "error": f"Error during MCP call: {e}",
            "data": None
        }


# Helper functions to convert to DataFrame
def create_census_dataframe(census_response):
    """
    Convert Census API response to pandas DataFrame.
    
    Args:
        census_response: Response from fetch_census_data_async()
    
    Returns:
        pandas.DataFrame or None if error
    """
    if not census_response["success"]:
        print(f"❌ Error: {census_response['error']}")
        return None
    
    data = census_response["data"]
    
    if isinstance(data, list) and len(data) > 0:
        # Data is already in the right format for DataFrame
        df = pd.DataFrame(data)
        
        # Convert numeric columns
        for col in df.columns:
            if col not in ['NAME', 'state', 'county', 'tract']:  # Keep geographic identifiers as strings
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df
    elif isinstance(data, str):
        print("❌ Received text response instead of structured data:")
        print(data[:500] + "..." if len(data) > 500 else data)
        return None
    else:
        print(f"❌ Unexpected data format received: {type(data)}")
        return None

# Convenience function that combines both steps
async def get_census_dataframe(
    dataset="acs5", 
    year=2022, 
    variables=["B01003_001E"], 
    geography="state", 
    geography_filter=None
):
    """
    One-step function to get Census data as DataFrame.
    
    Returns:
        pandas.DataFrame: Census data
    """
    response = await fetch_census_data_async(
        dataset=dataset,
        year=year,
        variables=variables,
        geography=geography,
        geography_filter=geography_filter
    )
    
    if response["success"]:
        print("✅ Data retrieved successfully!")
        if "summary" in response:
            # Print a cleaner summary
            summary_lines = response["summary"].split('\n')
            for line in summary_lines[:6]:  # Show first 6 lines of summary
                if line.strip():
                    print(line)
    
    return create_census_dataframe(response)

state_fips_to_name = {
    '01': 'Alabama', '02': 'Alaska', '04': 'Arizona', '05': 'Arkansas',
    '06': 'California', '08': 'Colorado', '09': 'Connecticut', '10': 'Delaware',
    '11': 'District of Columbia', '12': 'Florida', '13': 'Georgia', '15': 'Hawaii',
    '16': 'Idaho', '17': 'Illinois', '18': 'Indiana', '19': 'Iowa',
    '20': 'Kansas', '21': 'Kentucky', '22': 'Louisiana', '23': 'Maine',
    '24': 'Maryland', '25': 'Massachusetts', '26': 'Michigan', '27': 'Minnesota',
    '28': 'Mississippi', '29': 'Missouri', '30': 'Montana', '31': 'Nebraska',
    '32': 'Nevada', '33': 'New Hampshire', '34': 'New Jersey', '35': 'New Mexico',
    '36': 'New York', '37': 'North Carolina', '38': 'North Dakota', '39': 'Ohio',
    '40': 'Oklahoma', '41': 'Oregon', '42': 'Pennsylvania', '44': 'Rhode Island',
    '45': 'South Carolina', '46': 'South Dakota', '47': 'Tennessee', '48': 'Texas',
    '49': 'Utah', '50': 'Vermont', '51': 'Virginia', '53': 'Washington',
    '54': 'West Virginia', '55': 'Wisconsin', '56': 'Wyoming', '72': 'Puerto Rico'
}


## Some usage examples

In [2]:
# Example usage functions for Jupyter
async def get_state_population():
    """Get population by state."""
    df =  await get_census_dataframe(
        dataset="acs5",
        year=2022,
        variables=["B01003_001E"],  # Total population
        geography="state"
    )
    df['state_name'] = df['state'].map(state_fips_to_name)
    return df

df = await get_state_population()
df.head()


✅ Data retrieved successfully!
US Census Data Retrieved Successfully
Dataset: ('acs/acs5', 'American Community Survey 5-Year') (2022)
Geography: State
Variables: B01003_001E
Records: 52


,B01003_001E,state,state_name
0,5028092,01,Alabama
1,734821,02,Alaska
2,7172282,04,Arizona
3,3018669,05,Arkansas
4,39356104,06,California


In [16]:
async def get_california_counties():
    """Get population for California counties."""
    return await get_census_dataframe(
        dataset="acs5",
        year=2022,
        variables=["B01003_001E"],  # Total population
        geography="county",
        geography_filter={"state": "06"}  # California FIPS code
    )

df = await get_california_counties()
df.head()

✅ Data retrieved successfully!
US Census Data Retrieved Successfully
Dataset: ('acs/acs5', 'American Community Survey 5-Year') (2022)
Geography: County
Variables: B01003_001E
Records: 58


,B01003_001E,state,county
0,1663823,06,001
1,1515,06,003
2,40577,06,005
3,213605,06,007
4,45674,06,009


In [14]:
async def get_demographic_data():
    """Get multiple demographic variables by state."""
    df =  await get_census_dataframe(
        dataset="acs5",
        year=2022,
        variables=[
            "B01003_001E",  # Total population
            "B19013_001E",  # Median household income
            "B25077_001E"   # Median home value
        ],
        geography="state"
    )
    df['state_name'] = df['state'].map(state_fips_to_name)
    return df
df = await get_demographic_data()
df.head()

✅ Data retrieved successfully!
US Census Data Retrieved Successfully
Dataset: ('acs/acs5', 'American Community Survey 5-Year') (2022)
Geography: State
Variables: B01003_001E, B19013_001E, B25077_001E
Records: 52


,B01003_001E,B19013_001E,B25077_001E,state,state_name
0,5028092,59609,179400,01,Alabama
1,734821,86370,318000,02,Alaska
2,7172282,72581,321400,04,Arizona
3,3018669,56335,162400,05,Arkansas
4,39356104,91905,659300,06,California


In [6]:
# Example:  get the populations of various races in each state
async def get_racial_demographics():
    df =  await get_census_dataframe(
        dataset="acs5",
        year=2022,
        variables=[
            "B02001_002E",  # White alone
            "B02001_003E",  # Black or African American alone
            "B02001_004E",  # American Indian and Alaska Native alone
            "B02001_005E",  # Asian alone
            "B02001_006E",  # Native Hawaiian and Other Pacific Islander alone
            "B02001_007E",  # Some other race alone
        ],
        geography="state"
    )
    df['state_name'] = df['state'].map(state_fips_to_name)
    return df   
df = await get_racial_demographics()
df.head()

✅ Data retrieved successfully!
US Census Data Retrieved Successfully
Dataset: ('acs/acs5', 'American Community Survey 5-Year') (2022)
Geography: State
Variables: B02001_002E, B02001_003E, B02001_004E, B02001_005E, B02001_006E, B02001_007E
Records: 52


,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,state,state_name
0,3329012,1326341,21122,69808,2253,93924,01,Alabama
1,450472,23395,104957,47464,11209,14597,02,Alaska
2,4781702,327077,297590,240642,14116,549361,04,Arizona
3,2193348,456693,16840,47413,11117,89782,05,Arkansas
4,18943660,2202587,394188,5949136,150531,6388999,06,California


# Census Data Agent (that uses us_census_tool in MCP server)

In [2]:
import requests

def ask_census(question):
    """Ask the Census Data Agent a question"""
    payload = {
        "agent_name": "census_data_agent",
        "input": [{"parts": [{"content": question}]}]
    }
    
    response = requests.post(
        "http://localhost:8001/runs",
        json=payload,
        timeout=60
    )
    
    if response.status_code == 200:
        data = response.json()
        if data.get("status") == "completed":
            output = data.get("output", [])
            if output and output[0].get("parts"):
                return output[0]["parts"][0]["content"]
        return f"Run status: {data.get('status')}"
    else:
        return f"Error {response.status_code}: {response.text}"

# Example usage:
print(ask_census("What is the population of California?"))
print(ask_census("Which state has the highest median income?"))
print(ask_census("Which state has the largest proportion of Black Americans?"))

The population of California is 39,356,104 residents.
District of Columbia has the highest median household income at $101,722.
District of Columbia has the highest proportion of Black or African American. Of the 670,587 residents, 297,101 are Black or African American, constituting 44.3% of the population.
